# GSM8K Pipeline — Version robuste (sans PEFT)
T5-small full fine-tune, 5 époques, GPU T4 (~40 min)

In [ ]:
!pip install -q torch transformers datasets accelerate sentencepiece

In [ ]:
import sys, re, json, math, time, torch
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq)
print('✓ Imports')

In [ ]:
# Codec ψ minimal
import numpy as np
PHI = (1+5**0.5)/2; HALF_PI = 3.14159/2; PI = 3.14159; ZERO = 0.0
CODE_MAP = {'ADD':3,'SUBTRACT':1,'MULTIPLY':2,'DIVIDE':5}

def encoder_ops(ops):
    frames = []; variables = {}; vc = 0
    for op in ops:
        on = op.get('op','').upper(); vc += 1; vn = f'e{vc}'
        if on == 'INIT':
            try: v = float(op.get('value',0))
            except: v = 0.0
            variables[vn] = v
            frames.append({'code':4,'amp':abs(v),'phase':0.0 if v>=0 else PI,'op':'INIT','var':vn,'value':v})
            continue
        if on == 'QUERY': continue
        # Dernière variable modifiée
        sv = list(variables.values())[-1] if variables else 0.0
        for k in ('value','multiplier','divisor'):
            raw = op.get(k)
            if isinstance(raw,(int,float)):
                opd = float(raw); break
        else: continue
        cd = CODE_MAP.get(on,3)
        if on == 'ADD': nv = sv + opd; ph = ZERO
        elif on == 'SUBTRACT': nv = sv - opd; ph = PI
        elif on == 'MULTIPLY': nv = sv * opd; ph = ZERO
        elif on == 'DIVIDE': nv = sv / opd if opd else sv; ph = -HALF_PI
        else: nv = sv * opd; ph = ZERO
        variables[vn] = nv
        frames.append({'code':cd,'amp':1.0,'phase':HALF_PI,'op':on,'var':vn,'value':None})
        d = abs(nv - sv)
        frames.append({'code':cd,'amp':d if d>1e-9 else 1.0,'phase':ph,'op':on,'var':vn,'value':nv})
    return frames

def decoder_trames(frames):
    z = 0.0+0.0j; fv = None
    for f in frames:
        z += f['amp']*np.exp(1j*f['phase'])
        if f.get('value') is not None: fv = f['value']
    return float(z.real) if fv is None else fv
print('✓ Codec ψ')

In [ ]:
# Parseur d'annotations
ANOT_RE = re.compile(r'<<([^>]+)>>')
OP_MAP = {'+':'ADD','-':'SUBTRACT','*':'MULTIPLY','/':'DIVIDE'}
def _nettoyer(e):
    e = e.replace('(','').replace(')','')
    e = re.sub(r'--','+',e); e = re.sub(r'\+-','-',e); e = re.sub(r'-\+','-',e)
    return e
def anot2ops(answer):
    ops = []; chain = None
    for m in ANOT_RE.finditer(answer):
        expr = m.group(1)
        if '=' not in expr: continue
        ce, rs = expr.split('=',1)
        try: result = float(rs)
        except: continue
        clean = _nettoyer(ce)
        if clean.startswith('+'): clean = clean[1:]; chain = chain or 0.0
        neg = 1.0
        if clean.startswith('-'): clean = clean[1:]; neg = -1.0
        tokens = re.findall(r'[+\-*/]|\d+(?:\.\d+)?', clean)
        if not tokens: continue
        try: cur = float(tokens[0])*neg
        except: continue
        if chain is None or abs(cur-chain)>1e-9:
            ops.append({'op':'INIT','value':cur}); chain = cur
        i = 1
        while i+1 <= len(tokens)-1:
            if i+1 >= len(tokens): break
            op = tokens[i]; ns = tokens[i+1]
            if op not in OP_MAP: break
            try: nxt = float(ns)
            except: break
            m = OP_MAP[op]
            if m=='ADD': ops.append({'op':'ADD','value':nxt}); cur += nxt
            elif m=='SUBTRACT': ops.append({'op':'SUBTRACT','value':nxt}); cur -= nxt
            elif m=='MULTIPLY': ops.append({'op':'MULTIPLY','multiplier':nxt}); cur *= nxt
            elif m=='DIVIDE': ops.append({'op':'DIVIDE','divisor':nxt}); cur = cur/nxt if nxt else cur
            i += 2
        chain = result
    return ops
def reponse_finale(answer):
    m = re.search(r'####\s*(-?\d+(?:\.\d+)?)', answer)
    return float(m.group(1)) if m else None
def ops2texte(ops):
    parts = []
    for o in ops:
        if o['op']=='INIT': parts.append(f"INIT({o['value']})")
        elif o['op']=='ADD': parts.append(f"ADD({o['value']})")
        elif o['op']=='SUBTRACT': parts.append(f"SUB({o['value']})")
        elif o['op']=='MULTIPLY': parts.append(f"MUL({o['multiplier']})")
        elif o['op']=='DIVIDE': parts.append(f"DIV({o['divisor']})")
    return ' '.join(parts)
print('✓ Parseur')

In [ ]:
print('📦 Construction dataset...')
train = load_dataset('gsm8k','main',split='train')
test = load_dataset('gsm8k','main',split='test')
exemples = []
for item in train:
    exp = reponse_finale(item['answer'])
    if exp is None: continue
    ops = anot2ops(item['answer'])
    if not ops: continue
    try:
        got = decoder_trames(encoder_ops(ops))
        if got is None or abs(got-exp)>1e-6: continue
    except: continue
    exemples.append({'input':item['question'],'target':ops2texte(ops)})
print(f'Dataset : {len(exemples)} paires')
# Gold score
ok_gold = 0
for item in test:
    exp = reponse_finale(item['answer'])
    if exp is None: continue
    ops = anot2ops(item['answer'])
    if not ops: continue
    try:
        got = decoder_trames(encoder_ops(ops))
        ok_gold += got is not None and abs(got-exp)<1e-6
    except: continue
print(f'Gold (codec+annotations) : {ok_gold}/{len(test)} ({100*ok_gold/len(test):.1f}%)')

In [ ]:
# ── Génération transvertical + pré-entraînement ──
print("🌍 Génération dataset transvertical...")
import random; random.seed(42)
DOMAINES = ["maths","droit","medecine","logique","eco","physique","quotidien"]
NOMS = {"maths":["apples","books","dollars","candies","pencils","oranges","tickets"],
  "droit":["damages","penalties","clauses","claims","fees","fines"],
  "medecine":["milliliters","beats","milligrams","degrees","units","cells","drops","grams"],
  "logique":["premises","inferences","propositions","cases","instances","arguments"],
  "eco":["dollars","euros","shares","bonds","assets","revenues","costs"],
  "physique":["meters","seconds","grams","liters","joules","watts","volts"],
  "quotidien":["apples","cookies","cups","eggs","flowers","liters","tickets","books"]}
MOTIFS = {
  "INIT": {"maths":["has","starts with","buys","collects","receives","finds"],
    "droit":["files","claims","demands","seeks","requests","receives"],
    "medecine":["presents with","has","weighs","measures","shows","exhibits"],
    "logique":["assumes","posits","states","defines","proposes","asserts"],
    "eco":["invests","spends","budgets","allocates","earns","reports"],
    "physique":["measures","records","observes","calculates","reads","detects"],
    "quotidien":["has","buys","prepares","makes","bakes","cooks","grows"]},
  "SUB": {"maths":["gives away","loses","spends","sells","removes","eats","breaks"],
    "droit":["deducts","excludes","subtracts","waives","reduces by","lowers by"],
    "medecine":["the fever drops by","the patient loses","the count decreases by","symptoms improve by"],
    "logique":["is not the case","excludes","contradicts","negates","refutes","invalidates"],
    "eco":["loses","spends","incurs","pays","depreciates by","writes off"],
    "physique":["loses","dissipates","decays by","decreases by","cools by","slows by"],
    "quotidien":["gives away","eats","drinks","uses","spends","breaks","loses"]},
  "ADD": {"maths":["buys","gains","finds","receives","earns","collects","adds"],
    "droit":["adds to the settlement","includes","compensates","awards additional","grants","imposes"],
    "medecine":["gains weight","the count increases by","the heart rate rises by","symptoms worsen by"],
    "logique":["and additionally","combined with","together with","in conjunction with","alongside"],
    "eco":["earns","gains","receives","adds","accrues","generates revenue of","collects"],
    "physique":["gains","absorbs","increases by","accumulates","stores","charges","heats up by"],
    "quotidien":["buys","finds","receives","adds","picks up","gathers","collects","grows"]},
  "MUL": {"maths":["each has","times","per","for every","twice","three times","each of"],
    "droit":["for each violation","multiplied by the penalty","per article","for every instance","per defendant"],
    "medecine":["per dose","for each kilogram","per day","per patient","per session","for every hour"],
    "logique":["for every instance","in all cases","for each","applies to all","universally","for any"],
    "eco":["times the rate","per unit","for each item","per share","each unit costs","per transaction"],
    "physique":["per second","per meter","per kilogram","per hour","per unit volume","per degree"],
    "quotidien":["each","per","for every","per person","each of the","every","apiece","doubles"]},
  "DIV": {"maths":["split among","divided by","per person","each of","shared between","half of","quarter of"],
    "droit":["divided among the heirs","shared between parties","apportioned","split between plaintiffs","per capita"],
    "medecine":["per patient","divided in doses","per session","split into","divided by body weight","per kilogram"],
    "logique":["applies to each","distributed over","divided among","per instance","for each case","half of","third of"],
    "eco":["divided among","per share","per unit","split between","each investor gets","per partner","per capita"],
    "physique":["per unit","per meter","per kilogram","divided by","per second","per hour","per degree"],
    "quotidien":["split among","divided by","per person","each of","shared between","each gets","half of","quarter of"]},
}
GABARITS = [
  ["INIT","A","SUBTRACT","B"],
  ["INIT","A","ADD","B"],
  ["INIT","A","MULTIPLY","B"],
  ["INIT","A","DIVIDE","B"],
  ["INIT","A","MULTIPLY","B","SUBTRACT","C"],
  ["INIT","A","MULTIPLY","B","ADD","C"],
  ["INIT","A","MULTIPLY","B","DIVIDE","C"],
  ["INIT","A","MULTIPLY","B","ADD","C","DIVIDE","D"],
  ["INIT","A","MULTIPLY","B","SUBTRACT","C","DIVIDE","D","MULTIPLY","E"],
  ["INIT","A","MULTIPLY","B","INIT","C","SUBTRACT","D"],
  ["INIT","A","MULTIPLY","B","INIT","C","MULTIPLY","D","ADD","E"],
]
OPNAMES = {"INIT":"INIT","SUBTRACT":"SUB","ADD":"ADD","MULTIPLY":"MUL","DIVIDE":"DIV"}
tv_ex = []
for _ in range(10000):
  g = random.choice(GABARITS); d = random.choice(DOMAINES)
  vals = {"A":random.randint(1,100),"B":random.randint(1,20),"C":random.randint(1,15),"D":random.randint(1,10),"E":random.randint(1,8)}
  phrases = [random.choice(["In the","At the","During"])+" "+random.choice(["shop","market","lab","court","clinic","kitchen","field"])+","]
  for i in range(0,len(g),2):
    op, var = g[i], g[i+1]; v = vals[var]
    if op=="INIT": m=random.choice(MOTIFS["INIT"][d]); n=random.choice(NOMS[d]); phrases.append(f"{m} {v} {n}")
    elif op=="SUBTRACT": m=random.choice(MOTIFS["SUB"][d]); n=random.choice(NOMS[d]); phrases.append(f"{m} {v} {n}")
    elif op=="ADD": m=random.choice(MOTIFS["ADD"][d]); n=random.choice(NOMS[d]); phrases.append(f"{m} {v} {n}")
    elif op=="MULTIPLY": m=random.choice(MOTIFS["MUL"][d]); n=random.choice(NOMS[d]); phrases.append(f"{m} {v} {n}")
    elif op=="DIVIDE": m=random.choice(MOTIFS["DIV"][d]); n=random.choice(NOMS[d]); phrases.append(f"{m} {v} {n}")
  texte = ", ".join(phrases)+"."
  cible = []
  for i in range(0,len(g),2):
    op, var = g[i], g[i+1]; cible.append(f"{OPNAMES[op]}({vals[var]})")
  tv_ex.append({"input":texte,"target":" ".join(cible)})
print(f"✓ {len(tv_ex)} exemples transvertiaux générés")

# Pré-entraînement transvertical
print("\n🌍 1. Pré-entraînement transvertical (10k ex, 3 époques)...")
tv_ds = Dataset.from_list(tv_ex)
tv_split = tv_ds.train_test_split(test_size=0.05, seed=42)
def tv_tok(b):
  inp = tok(b["input"], max_length=256, truncation=True, padding=False)
  tgt = tok(b["target"], max_length=64, truncation=True, padding=False)
  inp["labels"] = tgt["input_ids"]; return inp
cols = tv_split["train"].column_names
tv_train = tv_split["train"].map(tv_tok, batched=True, remove_columns=cols)
tv_val = tv_split["test"].map(tv_tok, batched=True, remove_columns=cols)
tv_args = TrainingArguments(
  output_dir="/kaggle/working/t5_tv",
  num_train_epochs=3,
  per_device_train_batch_size=16,
  gradient_accumulation_steps=2,
  learning_rate=3e-4, warmup_ratio=0.1,
  logging_steps=50, eval_strategy="epoch", save_strategy="epoch",
  load_best_model_at_end=True, fp16=True, report_to="none",
  dataloader_num_workers=2, remove_unused_columns=False,
)
tv_trainer = Trainer(model=model, args=tv_args, train_dataset=tv_train, eval_dataset=tv_val,
  data_collator=DataCollatorForSeq2Seq(tok, model=model, padding=True), processing_class=tok)
tv_trainer.train()
print("✓ Pré-entraînement transvertical terminé")
print("\n📊 2. Fine-tuning sur GSM8K...")


In [ ]:
print("📊 2. Fine-tuning sur GSM8K (5 époques)...")
tok = AutoTokenizer.from_pretrained('google/flan-t5-small')
ds = Dataset.from_list(exemples)
split = ds.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split['train'], split['test']
def tok_fn(b):
    inp = tok(['translate to operations: '+t for t in b['input']], max_length=384, truncation=True, padding=False)
    tgt = tok(b['target'], max_length=128, truncation=True, padding=False)
    inp['labels'] = tgt['input_ids']; return inp
cols = train_ds.column_names
train_t = train_ds.map(tok_fn, batched=True, remove_columns=cols)
val_t = val_ds.map(tok_fn, batched=True, remove_columns=cols)
args = TrainingArguments(
    output_dir='/kaggle/working/t5_gsm8k',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=True,
    report_to='none',
    dataloader_num_workers=2,
    
)
trainer = Trainer(model=model, args=args, train_dataset=train_t, eval_dataset=val_t,
    data_collator=DataCollatorForSeq2Seq(tok, model=model, padding=True), processing_class=tok)
t0 = time.time()
trainer.train()
print(f'✓ Terminé en {(time.time()-t0)/60:.1f} min')
final_path = '/kaggle/working/t5_gsm8k_final'
model.save_pretrained(final_path)
tok.save_pretrained(final_path)
print(f'✓ Modèle sauvegardé : {final_path}')

In [ ]:
print('\n📊 Évaluation...')
model.eval()
if torch.cuda.is_available(): model = model.cuda()
def ops2seq(pred):
    OM = {'MUL':'MULTIPLY','SUB':'SUBTRACT','ADD':'ADD','DIV':'DIVIDE','INIT':'INIT'}
    ops = []
    for token in pred.replace('\n',' ').split():
        m = re.match(r'(INIT|MUL|SUB|ADD|DIV)\(([^)]+)\)', token.strip())
        if not m: continue
        op, v = m.group(1), m.group(2)
        try: v = float(v)
        except: continue
        mapped = OM.get(op)
        if not mapped: continue
        if mapped=='INIT': ops.append({'op':'INIT','value':v})
        elif mapped=='MULTIPLY': ops.append({'op':'MULTIPLY','multiplier':v})
        elif mapped=='DIVIDE': ops.append({'op':'DIVIDE','divisor':v})
        elif mapped=='SUBTRACT': ops.append({'op':'SUBTRACT','value':v})
        elif mapped=='ADD': ops.append({'op':'ADD','value':v})
    return ops
ok_model = 0
for item in test:
    exp = reponse_finale(item['answer'])
    if exp is None: continue
    inp = 'translate to operations: '+item['question']
    inputs = tok(inp, return_tensors='pt', max_length=384, truncation=True)
    if torch.cuda.is_available(): inputs = {k:v.cuda() for k,v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, num_beams=1)
    pred = tok.decode(out[0], skip_special_tokens=True)
    ops = ops2seq(pred)
    if not ops: continue
    try:
        got = decoder_trames(encoder_ops(ops))
        if got is not None and abs(got-exp)<1e-6: ok_model += 1
    except: continue
print(f'Score T5+codec : {ok_model}/{len(test)} ({100*ok_model/len(test):.1f}%)')
print(f'Score gold     : {ok_gold}/{len(test)} ({100*ok_gold/len(test):.1f}%)')
import json
with open('/kaggle/working/results.json','w') as f:
    json.dump({'gold':ok_gold/len(test),'model':ok_model/len(test)},f)
print('\n✅ Résultats dans /kaggle/working/results.json')
print('✅ Modèle dans /kaggle/working/t5_gsm8k_final/')